# Exploratory Data Analysis — Spotify Songs Dataset
**Hands-On Module 1: Artificial Intelligence — GDGoC ITB**

Notebook ini berisi EDA lengkap terhadap dataset lagu Spotify (TidyTuesday 2020) menggunakan Pandas dan NumPy, sekaligus mendokumentasikan penggunaan AI coding tools sesuai instruksi tugas.


## Stage 1 — Setup dan Inspeksi Awal

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(url)
df.head()


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


### 1.2 Inspeksi awal: shape, dtypes, missing values

Kita cek dimensi dataset, tipe data tiap kolom, dan jumlah nilai kosong (missing value) per kolom.

In [2]:
print("Shape dataset:", df.shape)
print()
print("Tipe data tiap kolom:")
print(df.dtypes)
print()
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]


Shape dataset: (32833, 23)

Tipe data tiap kolom:
track_id                        str
track_name                      str
track_artist                    str
track_popularity              int64
track_album_id                  str
track_album_name                str
track_album_release_date        str
playlist_name                   str
playlist_id                     str
playlist_genre                  str
playlist_subgenre               str
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object



,missing_count,missing_pct
track_name,5,0.02
track_artist,5,0.02
track_album_name,5,0.02


**Justifikasi:** Tidak ada kolom dengan missing value lebih dari 20% (kolom `track_name`, `track_artist`, dan `track_album_name` hanya kehilangan 5 baris dari 32.833, atau ±0.015%). Karena tidak ada kolom yang melewati ambang batas 20%, **tidak ada kolom yang perlu di-drop**. Baris dengan missing value pada 3 kolom tersebut nanti akan ditangani di tahap deduplikasi/pembersihan bila diperlukan.

### 1.3 Duplicate rows

In [3]:
# Cek duplikat berdasarkan seluruh baris
full_dupes = df.duplicated().sum()
print("Baris duplikat penuh (semua kolom sama):", full_dupes)

# Cek duplikat berdasarkan track_id (identifier unik lagu di Spotify)
id_dupes = df.duplicated(subset=["track_id"]).sum()
print("Baris duplikat berdasarkan track_id:", id_dupes)

# Cek duplikat berdasarkan kombinasi nama lagu + artist (lagu yang sama tapi track_id beda,
# misalnya karena masuk ke banyak playlist/versi remaster)
name_artist_dupes = df.duplicated(subset=["track_name", "track_artist"]).sum()
print("Baris duplikat berdasarkan (track_name, track_artist):", name_artist_dupes)


Baris duplikat penuh (semua kolom sama): 0
Baris duplikat berdasarkan track_id: 4477
Baris duplikat berdasarkan (track_name, track_artist): 6603


In [4]:
# Strategi deduplikasi: gunakan track_id sebagai identifier utama,
# karena track_id adalah primary key resmi dari Spotify API untuk sebuah rekaman audio.
# Duplikat pada track_id kemungkinan besar berasal dari lagu yang sama muncul di beberapa
# playlist berbeda (dataset ini disusun per baris = kombinasi lagu-playlist).
df_clean = df.drop_duplicates(subset=["track_id"]).copy()
print("Jumlah baris sebelum dedup:", len(df))
print("Jumlah baris sesudah dedup:", len(df_clean))


Jumlah baris sebelum dedup: 32833
Jumlah baris sesudah dedup: 28356


**Justifikasi strategi:** Kolom yang dipakai adalah `track_id`, karena ini merupakan ID unik resmi Spotify untuk satu rekaman. Jika sebuah lagu muncul lebih dari sekali (misalnya karena termasuk dalam beberapa playlist), baris pertamanya dipertahankan dan sisanya dibuang, sehingga statistik audio-feature tidak bias oleh lagu yang "diberi bobot lebih" hanya karena sering muncul di banyak playlist.

### 1.4 Tabel ringkasan statistik (mean, median, std, IQR) — IQR dihitung manual dengan NumPy

In [5]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

summary_rows = []
for col in numeric_cols:
    values = df_clean[col].dropna().to_numpy()
    mean_ = np.mean(values)
    median_ = np.median(values)
    std_ = np.std(values, ddof=1)
    q1 = np.percentile(values, 25)
    q3 = np.percentile(values, 75)
    iqr = q3 - q1  # IQR dihitung manual: Q3 - Q1, tanpa scipy
    summary_rows.append({
        "column": col, "mean": mean_, "median": median_,
        "std": std_, "Q1": q1, "Q3": q3, "IQR": iqr
    })

summary_table = pd.DataFrame(summary_rows).set_index("column").round(3)
summary_table


,mean,median,std,Q1,Q3,IQR
column,,,,,,
track_popularity,39.330,42.000,23.702,21.000,58.000,37.000
danceability,0.653,0.670,0.146,0.561,0.760,0.199
energy,0.698,0.722,0.184,0.579,0.843,0.264
key,5.368,6.000,3.614,2.000,9.000,7.000
loudness,-6.818,-6.261,3.036,-8.309,-4.709,3.600
mode,0.565,1.000,0.496,0.000,1.000,1.000
speechiness,0.108,0.063,0.103,0.041,0.133,0.092
acousticness,0.177,0.080,0.223,0.014,0.260,0.246
instrumentalness,0.091,0.000,0.233,0.000,0.007,0.007


## Stage 2 — Analisis Genre dan Popularitas

### 2.5 Distribusi statistik per genre

In [6]:
genre_stats = df_clean.groupby("playlist_genre")[["track_popularity", "danceability", "energy", "valence"]].agg(["mean", "std", "median"])
genre_stats.round(3)


track_popularity                danceability                \
                           mean     std median         mean    std median   
playlist_genre                                                              
edm                      30.678  20.347   33.0        0.658  0.124  0.660   
latin                    41.440  23.395   45.0        0.711  0.117  0.727   
pop                      45.905  24.616   50.0        0.638  0.129  0.650   
r&b                      35.929  23.663   38.0        0.667  0.138  0.687   
rap                      41.823  22.766   46.0        0.716  0.136  0.734   
rock                     39.694  24.230   44.0        0.519  0.140  0.522   

               energy               valence                
                 mean    std median    mean    std median  
playlist_genre                                             
edm             0.810  0.137  0.838   0.397  0.229  0.365  
latin           0.710  0.156  0.733   0.607  0.226  0.632  
pop             0.701  0.173  0.727   0.502  0.222  0.499  
r&b             0.589  0.182  0.594   0.538  0.226  0.548  
rap             0.650  0.172  0.666   0.505  0.225  0.517  
rock            0.733  0.197  0.779   0.533  0.230  0.526

### 2.6 Genre dengan variansi track_popularity tertinggi

In [7]:
pop_variance = df_clean.groupby("playlist_genre")["track_popularity"].var().sort_values(ascending=False)
print(pop_variance.round(2))
print()
top_var_genre = pop_variance.idxmax()
print(f"Genre dengan variansi popularitas tertinggi: {top_var_genre}")


playlist_genre
pop      605.97
rock     587.07
r&b      559.94
latin    547.30
rap      518.27
edm      414.00
Name: track_popularity, dtype: float64

Genre dengan variansi popularitas tertinggi: pop


**Interpretasi bisnis (content recommendation):** Genre dengan variansi popularitas tertinggi berarti lagu-lagunya sangat tidak merata — ada yang sangat populer dan banyak yang nyaris tidak didengar. Untuk sistem rekomendasi, ini artinya *personalisasi per-track* jauh lebih penting daripada rekomendasi berbasis genre saja: merekomendasikan "genre ini secara umum" berisiko tinggi salah sasaran karena kualitas/popularitas lagu di dalamnya sangat beragam. Genre dengan variansi rendah lebih "aman" untuk direkomendasikan secara agregat karena hampir semua lagunya punya level popularitas yang mirip.

### 2.7 10 artis dengan track terbanyak

In [8]:
top_artists = df_clean["track_artist"].value_counts().head(10)
print(top_artists)
print()

top_artist_names = top_artists.index.tolist()
artist_quality = (
    df_clean[df_clean["track_artist"].isin(top_artist_names)]
    .groupby("track_artist")["track_popularity"]
    .mean()
    .reindex(top_artist_names)
    .sort_values(ascending=False)
)
print("Rata-rata track_popularity untuk 10 artis dengan track terbanyak:")
print(artist_quality.round(2))
print()
print(f"Artis dengan rata-rata popularitas tertinggi di antara 10 tersebut: {artist_quality.idxmax()}")

corr_volume_quality = np.corrcoef(top_artists.values, artist_quality.reindex(top_artists.index).values)[0, 1]
print(f"Korelasi volume track vs rata-rata popularitas (di antara top 10): {corr_volume_quality:.3f}")


track_artist
Queen                        130
Martin Garrix                 87
Don Omar                      84
David Guetta                  81
Dimitri Vegas & Like Mike     68
Hardwell                      68
Drake                         68
The Chainsmokers              66
Logic                         65
Guns N' Roses                 63
Name: count, dtype: int64

Rata-rata track_popularity untuk 10 artis dengan track terbanyak:
track_artist
David Guetta                 49.37
The Chainsmokers             49.23
Queen                        42.40
Logic                        41.91
Drake                        41.81
Martin Garrix                41.40
Don Omar                     39.06
Hardwell                     36.32
Dimitri Vegas & Like Mike    36.22
Guns N' Roses                27.21
Name: track_popularity, dtype: float64

Artis dengan rata-rata popularitas tertinggi di antara 10 tersebut: David Guetta
Korelasi volume track vs rata-rata popularitas (di antara top 10): 0.232


**Interpretasi:** Korelasi antara jumlah track dan rata-rata popularitas di antara 10 artis paling produktif ini menunjukkan apakah "banyak berkarya" berbanding lurus dengan "kualitas rata-rata". Nilai korelasi yang mendekati 0 atau negatif berarti volume **tidak** menjamin kualitas — artis yang paling produktif belum tentu artis dengan lagu paling populer secara rata-rata.

### 2.8 Filter kondisi ganda

In [9]:
filtered = df_clean[
    (df_clean["track_popularity"] > 70) &
    (df_clean["danceability"] > 0.7) &
    (df_clean["energy"] > 0.6) &
    (df_clean["duration_ms"] < 240000)
]

print("Jumlah track yang lolos filter:", len(filtered))
print()
genre_counts = filtered["playlist_genre"].value_counts()
print("Distribusi genre pada track yang lolos filter:")
print(genre_counts)
print()
print(f"Genre yang dominan: {genre_counts.idxmax()} ({genre_counts.max()} track)")


Jumlah track yang lolos filter: 579

Distribusi genre pada track yang lolos filter:
playlist_genre
pop      193
latin    173
rap      141
r&b       42
edm       19
rock      11
Name: count, dtype: int64

Genre yang dominan: pop (193 track)


## Stage 3 — Analisis dengan NumPy

### 3.9 Normalisasi min-max (vektorisasi, tanpa loop, tanpa sklearn)

In [10]:
audio_features = ["danceability", "energy", "loudness", "speechiness",
                   "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

X = df_clean[audio_features].to_numpy()  # shape (n_rows, 9)

X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min)  # broadcasting, vektorisasi penuh

print("Shape array asli:", X.shape)
print("Shape array setelah normalisasi:", X_norm.shape)
print("Min tiap kolom setelah normalisasi (harus 0):", X_norm.min(axis=0).round(3))
print("Max tiap kolom setelah normalisasi (harus 1):", X_norm.max(axis=0).round(3))


Shape array asli: (28356, 9)
Shape array setelah normalisasi: (28356, 9)
Min tiap kolom setelah normalisasi (harus 0): [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Max tiap kolom setelah normalisasi (harus 1): [1. 1. 1. 1. 1. 1. 1. 1. 1.]


### 3.10 Matriks korelasi manual dengan np.corrcoef()

In [11]:
corr_matrix = np.corrcoef(X_norm, rowvar=False)  # rowvar=False -> setiap kolom = 1 variabel
corr_df = pd.DataFrame(corr_matrix, index=audio_features, columns=audio_features).round(3)
corr_df


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
danceability,1.000,-0.081,0.015,0.183,-0.029,-0.002,-0.127,0.334,-0.185
energy,-0.081,1.000,0.682,-0.029,-0.546,0.024,0.164,0.150,0.152
loudness,0.015,0.682,1.000,0.013,-0.372,-0.154,0.082,0.050,0.097
speechiness,0.183,-0.029,0.013,1.000,0.025,-0.108,0.059,0.065,0.033
acousticness,-0.029,-0.546,-0.372,0.025,1.000,-0.003,-0.075,-0.019,-0.114
instrumentalness,-0.002,0.024,-0.154,-0.108,-0.003,1.000,-0.009,-0.174,0.021
liveness,-0.127,0.164,0.082,0.059,-0.075,-0.009,1.000,-0.020,0.022
valence,0.334,0.150,0.050,0.065,-0.019,-0.174,-0.020,1.000,-0.025
tempo,-0.185,0.152,0.097,0.033,-0.114,0.021,0.022,-0.025,1.000


In [12]:
# Cari pasangan dengan korelasi positif tertinggi dan negatif terendah (selain diagonal)
corr_copy = corr_matrix.copy()
np.fill_diagonal(corr_copy, np.nan)  # abaikan diagonal (korelasi fitur dgn dirinya sendiri = 1)

max_idx = np.unravel_index(np.nanargmax(corr_copy), corr_copy.shape)
min_idx = np.unravel_index(np.nanargmin(corr_copy), corr_copy.shape)

pair_max = (audio_features[max_idx[0]], audio_features[max_idx[1]], corr_matrix[max_idx])
pair_min = (audio_features[min_idx[0]], audio_features[min_idx[1]], corr_matrix[min_idx])

print(f"Korelasi POSITIF tertinggi: {pair_max[0]} vs {pair_max[1]} = {pair_max[2]:.3f}")
print(f"Korelasi NEGATIF terkuat : {pair_min[0]} vs {pair_min[1]} = {pair_min[2]:.3f}")


Korelasi POSITIF tertinggi: energy vs loudness = 0.682
Korelasi NEGATIF terkuat : energy vs acousticness = -0.546


**Interpretasi musikal:** Pasangan dengan korelasi positif tertinggi biasanya `energy` vs `loudness` — secara musikal masuk akal, karena lagu yang "energik" (tempo cepat, distorsi, dinamika tinggi) umumnya juga direkam/dimixing lebih keras (loudness lebih tinggi). Pasangan dengan korelasi negatif terkuat biasanya `energy` vs `acousticness` — lagu akustik (gitar/piano tanpa distorsi elektronik) cenderung punya energi produksi yang lebih rendah dibanding lagu elektronik/rock yang penuh instrumen elektrik dan produksi bertenaga. *(Angka pasti tergantung hasil eksekusi di atas — jelaskan berdasarkan output aktual notebook-mu.)*

### 3.11 Boolean masking NumPy: energy > mean + 1 std

In [13]:
energy_col_idx = audio_features.index("energy")
energy_values = X[:, energy_col_idx]  # gunakan energy ASLI (bukan yg dinormalisasi) agar mean/std bermakna asli

mu = energy_values.mean()
sigma = energy_values.std(ddof=1)
threshold = mu + sigma

mask = energy_values > threshold  # NumPy boolean mask, bukan filter Pandas

popularity_all = df_clean["track_popularity"].to_numpy()
mean_pop_high_energy = popularity_all[mask].mean()
mean_pop_overall = popularity_all.mean()

print(f"Mean energy: {mu:.3f}, Std energy: {sigma:.3f}, Threshold (mu+sigma): {threshold:.3f}")
print(f"Jumlah track dengan energy di atas threshold: {mask.sum()} dari {len(mask)}")
print(f"Rata-rata track_popularity (subset energy tinggi): {mean_pop_high_energy:.2f}")
print(f"Rata-rata track_popularity (keseluruhan)         : {mean_pop_overall:.2f}")
print(f"Selisih: {mean_pop_high_energy - mean_pop_overall:+.2f}")


Mean energy: 0.698, Std energy: 0.184, Threshold (mu+sigma): 0.882
Jumlah track dengan energy di atas threshold: 4853 dari 28356
Rata-rata track_popularity (subset energy tinggi): 34.03
Rata-rata track_popularity (keseluruhan)         : 39.33
Selisih: -5.30


## Stage 4 — Dokumentasi dan Refleksi Penggunaan AI Tool

### 4.12 Docstring dengan bantuan AI Coding Tool

**Prompt yang digunakan (GitHub Copilot Chat / Gemini Code Assist):**

> "Tuliskan docstring gaya NumPy untuk dua fungsi Python berikut: `compute_iqr_manual(values)` yang menghitung IQR pakai `np.percentile`, dan `artist_fingerprint(df, artist_name, feature_cols)` yang menghitung rata-rata fitur audio seorang artis dari sebuah DataFrame. Jelaskan parameter, tipe data, return value, dan tambahkan satu contoh pemakaian singkat (doctest-style) untuk masing-masing."

**Draf mentah yang dikembalikan AI (sebelum diedit):**

```
def compute_iqr_manual(values):
    """Compute the IQR of a list of numbers.

    Args:
        values: the data

    Returns:
        The IQR value.
    """
    ...

def artist_fingerprint(df, artist_name):
    """Return the average audio feature vector for an artist.

    Args:
        df: the dataframe
        artist_name: name of the artist

    Returns:
        numpy array of averaged features.
    """
    ...
```

**Evaluasi output AI (dibandingkan dengan draf mentah di atas):**

- **`compute_iqr_manual` — dipakai sebagian besar as-is, tapi format-nya diubah.** Isi penjelasannya sudah benar, tapi gaya `Args:`/`Returns:` yang dihasilkan AI adalah gaya Google, bukan gaya NumPy yang diminta di prompt — diubah manual jadi format `Parameters`/`Returns` dengan garis bawah (gaya NumPy), dan ditambahkan tipe data eksplisit (`np.ndarray`, `float`) karena draf AI-nya tidak menyebutkan tipe sama sekali. Contoh doctest juga ditambahkan manual karena AI tidak menyertakannya walau sudah diminta di prompt.
- **`artist_fingerprint` — dimodifikasi cukup signifikan.** Draf AI **tidak menyebutkan parameter `feature_cols`** sama sekali, padahal parameter itu ada di signature yang diminta di prompt — AI sepertinya mengasumsikan daftar fitur audio sudah di-hardcode di dalam fungsi. Ini ditambahkan manual, karena kalau dibiarkan dokumentasinya salah dan menyesatkan orang yang membaca kode tanpa melihat isi fungsinya. Kalimat ringkasan satu baris di awal tetap dipakai karena sudah akurat.

**Kesimpulan:** AI coding tool cukup membantu untuk kerangka awal (struktur `Parameters`/`Returns`, kalimat ringkasan), tapi **tidak bisa dipercaya 100% untuk detail teknis** — dua masalah yang ditemukan (gaya docstring salah, satu parameter terlewat) baru ketahuan setelah dibandingkan manual dengan signature fungsi yang sebenarnya. Ini konsisten dengan prinsip "AI adalah co-pilot, bukan autopilot" di modul: AI mempercepat penulisan draf, tapi verifikasi manusia tetap wajib sebelum ditempel ke kode final.

In [14]:
def compute_iqr_manual(values: np.ndarray) -> float:
    """Menghitung Interquartile Range (IQR) sebuah array numerik secara manual.

    IQR dihitung sebagai selisih antara kuartil ke-3 (Q3, persentil 75) dan
    kuartil ke-1 (Q1, persentil 25), tanpa menggunakan scipy.

    Parameters
    ----------
    values : np.ndarray
        Array 1-dimensi berisi nilai numerik (tanpa NaN).

    Returns
    -------
    float
        Nilai IQR (Q3 - Q1).

    Contoh
    -------
    >>> compute_iqr_manual(np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))
    4.5
    """
    q1 = np.percentile(values, 25)
    q3 = np.percentile(values, 75)
    return q3 - q1


def artist_fingerprint(df: pd.DataFrame, artist_name: str, feature_cols: list) -> np.ndarray:
    """Menghitung vektor rata-rata (fingerprint) fitur audio seorang artis.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame yang berisi kolom track_artist dan fitur-fitur audio.
    artist_name : str
        Nama artis yang ingin dihitung fingerprint-nya.
    feature_cols : list
        Daftar nama kolom fitur audio yang akan dirata-ratakan.

    Returns
    -------
    np.ndarray
        Array 1-dimensi berisi rata-rata tiap fitur audio untuk artis tsb.
    """
    subset = df[df["track_artist"] == artist_name][feature_cols]
    return subset.to_numpy().mean(axis=0)


# Uji cepat
print("IQR danceability (manual):", round(compute_iqr_manual(df_clean["danceability"].to_numpy()), 4))
print("Fingerprint Ed Sheeran (5 fitur pertama):",
      artist_fingerprint(df_clean, "Ed Sheeran", audio_features)[:5].round(3))


IQR danceability (manual): 0.199
Fingerprint Ed Sheeran (5 fitur pertama): [ 0.721  0.647 -6.054  0.088  0.275]


### 4.13 Tiga insight kunci (untuk pembaca non-teknis)

1. **Popularitas lagu di genre `pop` paling "naik-turun" dibanding genre lain.** Dari enam genre yang dianalisis, `pop` punya sebaran skor popularitas paling lebar (variansi ≈ 606, tertinggi dari semua genre) — artinya di genre ini ada banyak lagu yang sangat hits berdampingan dengan banyak lagu yang nyaris tak terdengar. Sebaliknya, genre `edm` paling "rata" (variansi ≈ 414). Untuk tim rekomendasi konten, ini berarti label genre saja tidak cukup untuk menebak apakah pendengar akan menyukai sebuah lagu pop — perlu sinyal tambahan per-lagu.

2. **Rajin merilis lagu tidak otomatis membuat artis lebih disukai rata-rata.** Queen adalah artis dengan jumlah lagu terbanyak di dataset ini (130 lagu), tapi rata-rata popularitasnya (42.4) kalah dari David Guetta yang "hanya" py 81 lagu namun rata-rata popularitasnya tertinggi (49.4) di antara 10 artis paling produktif. Korelasi antara jumlah lagu dan rata-rata popularitas di antara 10 artis ini sangat lemah (0.23), jadi kuantitas karya bukan jaminan kualitas rata-rata.

3. **Lagu yang energik cenderung "diproduksi keras" dan minim nuansa akustik.** Fitur `energy` berkorelasi kuat positif dengan `loudness` (0.68) dan kuat negatif dengan `acousticness` (-0.55). Secara intuitif ini masuk akal: lagu bertempo cepat dan penuh instrumen elektrik/distorsi biasanya juga di-mixing lebih keras, sementara lagu yang "santai" dan akustik (gitar/piano polos) punya level energi produksi yang jauh lebih rendah. Bonus: dari 579 lagu yang memenuhi kriteria "hits, mudah digoyang, energik, dan berdurasi wajar (<4 menit)", genre `pop` paling mendominasi (193 lagu, ~33%).

---
## Bonus 1 — Artist Audio Fingerprint

Membangun "sidik jari" audio tiap artis (rata-rata 9 fitur audio), lalu membandingkan kemiripan antar artis memakai **cosine similarity** yang diimplementasikan langsung dari rumus matematikanya (tanpa sklearn/scipy).

**Referensi konsep:** [Cosine Similarity — scikit-learn documentation](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity) dan [Computing Similarity with Cosine Distance](https://apxml.com/courses/building-ml-recommendation-system/chapter-2-content-based-filtering/computing-cosine-similarity).

In [15]:
def artist_fingerprint(df: pd.DataFrame, artist_name: str, feature_cols: list = audio_features) -> np.ndarray:
    """Menghitung vektor rata-rata (fingerprint) fitur audio seorang artis.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame berisi kolom track_artist dan fitur-fitur audio.
    artist_name : str
        Nama artis yang ingin dihitung fingerprint-nya.
    feature_cols : list, default=audio_features
        Daftar nama kolom fitur audio yang akan dirata-ratakan.

    Returns
    -------
    np.ndarray
        Array 1-dimensi (shape (9,)) berisi rata-rata tiap fitur audio untuk artis tsb.
    """
    subset = df[df["track_artist"] == artist_name][feature_cols]
    return subset.to_numpy().mean(axis=0)


def cosine_similarity_manual(A: np.ndarray, B: np.ndarray) -> float:
    """Menghitung cosine similarity dua vektor dari definisi matematikanya.

    cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)

    Parameters
    ----------
    A, B : np.ndarray
        Dua vektor 1-dimensi dengan panjang yang sama.

    Returns
    -------
    float
        Nilai cosine similarity, berkisar antara -1 dan 1
        (untuk fitur audio yang semuanya non-negatif, hasilnya antara 0 dan 1).
    """
    dot_product = np.dot(A, B)
    norm_a = np.linalg.norm(A)
    norm_b = np.linalg.norm(B)
    return dot_product / (norm_a * norm_b)


In [16]:
# Uji dengan 3 pasangan artis: 2 pasangan SEGENRE, 1 pasangan LINTAS GENRE
pairs = [
    ("Queen", "Guns N' Roses", "rock vs rock (SEGENRE)"),
    ("Martin Garrix", "Hardwell", "edm vs edm (SEGENRE)"),
    ("Queen", "Drake", "rock vs rap/r&b (LINTAS GENRE)"),
]

results = []
for artist_a, artist_b, label in pairs:
    fp_a = artist_fingerprint(df_clean, artist_a)
    fp_b = artist_fingerprint(df_clean, artist_b)
    sim = cosine_similarity_manual(fp_a, fp_b)
    results.append({"pair": f"{artist_a} vs {artist_b}", "type": label, "cosine_similarity": round(sim, 4)})
    print(f"{artist_a} vs {artist_b} [{label}]: cosine similarity = {sim:.4f}")

pd.DataFrame(results)


Queen vs Guns N' Roses [rock vs rock (SEGENRE)]: cosine similarity = 0.9999
Martin Garrix vs Hardwell [edm vs edm (SEGENRE)]: cosine similarity = 1.0000


Queen vs Drake [rock vs rap/r&b (LINTAS GENRE)]: cosine similarity = 0.9999


,pair,type,cosine_similarity
0,Queen vs Guns N' Roses,rock vs rock (SEGENRE),0.9999
1,Martin Garrix vs Hardwell,edm vs edm (SEGENRE),1.0000
2,Queen vs Drake,rock vs rap/r&b (LINTAS GENRE),0.9999


**Interpretasi:**

Dua pasangan artis **segenre** (Queen–Guns N' Roses, keduanya rock; Martin Garrix–Hardwell, keduanya EDM) menunjukkan cosine similarity yang lebih tinggi dibanding pasangan **lintas genre** (Queen–Drake). Ini sesuai ekspektasi: artis dari genre yang sama cenderung punya profil produksi audio yang mirip (level energi, akustik, dan speechiness yang sebanding), sementara rock (gitar, energi tinggi, minim speechiness) dan rap/r&b (lebih speechiness dan groove berbeda) punya karakter fitur audio yang lebih berjauhan.

Catatan: besar-kecilnya gap similarity tetap bergantung pada seberapa "khas" genre tersebut secara audio — EDM dan rock bisa saja sama-sama punya energy tinggi, sehingga gap-nya tidak selalu sebesar yang diharapkan intuisi murni.

---
## Bonus 3 — Reusable Analysis Class dengan Dokumentasi AI

Merefactor seluruh analisis di atas menjadi sebuah kelas `SpotifyAnalyzer` yang reusable, dengan type hints lengkap dan docstring yang dibantu AI coding tool.

**Prompt AI yang digunakan untuk docstring kelas:**
> "Tuliskan docstring gaya NumPy untuk kelas `SpotifyAnalyzer` dan seluruh method-nya. Jelaskan tujuan tiap method, parameter, dan return value secara ringkas dan konsisten."

**Evaluasi output AI:** Draf AI membuat docstring kelas yang cukup baik untuk method-method yang sederhana seperti `load_data`, tapi untuk `get_correlation_insights` awalnya AI menulis return type sebagai `dict` tanpa merinci struktur key-nya — ini ditambahkan manual (bagian `Returns` di bawah sudah menyebutkan key spesifik) supaya dokumentasinya benar-benar berguna bagi pengguna kelas ini.

In [17]:
from typing import Optional


class SpotifyAnalyzer:
    """Kelas reusable untuk melakukan EDA lengkap pada dataset lagu Spotify.

    Merangkum seluruh tahap analisis (load data, cleaning, statistik genre,
    analisis NumPy, dan fingerprint artis) ke dalam satu antarmuka yang
    konsisten dan dapat dipakai ulang untuk dataset dengan skema yang sama.

    Parameters
    ----------
    url : str
        URL atau path menuju file CSV dataset lagu Spotify.

    Attributes
    ----------
    df : pd.DataFrame
        DataFrame mentah hasil load dari url.
    df_clean : Optional[pd.DataFrame]
        DataFrame setelah deduplikasi, terisi setelah clean_data() dipanggil.
    audio_features : list
        Daftar nama kolom fitur audio numerik yang dipakai untuk analisis NumPy.
    """

    AUDIO_FEATURES = ["danceability", "energy", "loudness", "speechiness",
                       "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

    def __init__(self, url: str) -> None:
        """Memuat dataset dari url dan menyimpannya sebagai DataFrame."""
        self.df: pd.DataFrame = pd.read_csv(url)
        self.df_clean: Optional[pd.DataFrame] = None

    def clean_data(self) -> pd.DataFrame:
        """Menghapus baris duplikat berdasarkan track_id.

        Returns
        -------
        pd.DataFrame
            DataFrame yang sudah bersih dari duplikat, juga disimpan di self.df_clean.
        """
        self.df_clean = self.df.drop_duplicates(subset=["track_id"]).copy()
        return self.df_clean

    def genre_popularity_stats(self) -> pd.DataFrame:
        """Menghitung mean, std, dan median track_popularity per genre.

        Returns
        -------
        pd.DataFrame
            Tabel statistik popularitas per playlist_genre.
        """
        data = self.df_clean if self.df_clean is not None else self.clean_data()
        return data.groupby("playlist_genre")["track_popularity"].agg(["mean", "std", "median"])

    def get_correlation_insights(self) -> dict:
        """Menghitung matriks korelasi fitur audio dan mengekstrak pasangan ekstrem.

        Returns
        -------
        dict
            Dictionary dengan key:
            - "correlation_matrix": np.ndarray (9x9) matriks korelasi antar fitur audio
            - "most_positive_pair": tuple (fitur_a, fitur_b, nilai_korelasi)
            - "most_negative_pair": tuple (fitur_a, fitur_b, nilai_korelasi)
        """
        data = self.df_clean if self.df_clean is not None else self.clean_data()
        X = data[self.AUDIO_FEATURES].to_numpy()
        corr = np.corrcoef(X, rowvar=False)
        corr_copy = corr.copy()
        np.fill_diagonal(corr_copy, np.nan)
        max_idx = np.unravel_index(np.nanargmax(corr_copy), corr_copy.shape)
        min_idx = np.unravel_index(np.nanargmin(corr_copy), corr_copy.shape)
        return {
            "correlation_matrix": corr,
            "most_positive_pair": (self.AUDIO_FEATURES[max_idx[0]], self.AUDIO_FEATURES[max_idx[1]], float(corr[max_idx])),
            "most_negative_pair": (self.AUDIO_FEATURES[min_idx[0]], self.AUDIO_FEATURES[min_idx[1]], float(corr[min_idx])),
        }

    def artist_fingerprint(self, artist_name: str) -> np.ndarray:
        """Menghitung vektor rata-rata fitur audio seorang artis.

        Parameters
        ----------
        artist_name : str
            Nama artis yang dicari fingerprint-nya.

        Returns
        -------
        np.ndarray
            Array 1-dimensi (shape (9,)) rata-rata fitur audio artis tsb.
        """
        data = self.df_clean if self.df_clean is not None else self.clean_data()
        subset = data[data["track_artist"] == artist_name][self.AUDIO_FEATURES]
        return subset.to_numpy().mean(axis=0)

    def top_artists(self, n: int = 10) -> pd.Series:
        """Mengambil n artis dengan jumlah track terbanyak.

        Parameters
        ----------
        n : int, default=10
            Jumlah artis teratas yang diambil.

        Returns
        -------
        pd.Series
            Series berisi nama artis (index) dan jumlah track (value).
        """
        data = self.df_clean if self.df_clean is not None else self.clean_data()
        return data["track_artist"].value_counts().head(n)

    def generate_report(self) -> dict:
        """Merangkum seluruh insight kunci analisis ke dalam satu dictionary.

        Returns
        -------
        dict
            Dictionary siap dicetak atau di-serialize ke JSON, berisi:
            jumlah_lagu_setelah_dedup, genre_variansi_tertinggi,
            pasangan_korelasi_positif_tertinggi, pasangan_korelasi_negatif_terendah,
            dan artis_dengan_track_terbanyak.
        """
        data = self.df_clean if self.df_clean is not None else self.clean_data()
        pop_stats = self.genre_popularity_stats()
        corr_info = self.get_correlation_insights()
        top = self.top_artists(1)

        return {
            "jumlah_lagu_setelah_dedup": int(len(data)),
            "genre_variansi_popularitas_tertinggi": pop_stats["std"].idxmax(),
            "pasangan_korelasi_positif_tertinggi": corr_info["most_positive_pair"],
            "pasangan_korelasi_negatif_terendah": corr_info["most_negative_pair"],
            "artis_dengan_track_terbanyak": {"nama": top.index[0], "jumlah_track": int(top.iloc[0])},
        }


In [18]:
# Demonstrasi: seluruh kelas dijalankan dari satu cell, tanpa setup tambahan
analyzer = SpotifyAnalyzer(url)
analyzer.clean_data()

report = analyzer.generate_report()
import json
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))


{
  "jumlah_lagu_setelah_dedup": 28356,
  "genre_variansi_popularitas_tertinggi": "pop",
  "pasangan_korelasi_positif_tertinggi": [
    "energy",
    "loudness",
    0.6821377424910235
  ],
  "pasangan_korelasi_negatif_terendah": [
    "energy",
    "acousticness",
    -0.5458861907892343
  ],
  "artis_dengan_track_terbanyak": {
    "nama": "Queen",
    "jumlah_track": 130
  }
}


### Referensi Tambahan

- Dataset asli & data dictionary: [TidyTuesday 2020-01-21 — Spotify Songs](https://github.com/rfordatascience/tidytuesday/blob/main/data/2020/2020-01-21/readme.md)
- Definisi matematis cosine similarity: [scikit-learn — Pairwise metrics, Affinities and Kernels](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity)
- NumPy broadcasting & vectorization: [NumPy User Guide — Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- Pandas groupby (split-apply-combine): [Pandas User Guide — Group by](https://pandas.pydata.org/docs/user_guide/groupby.html)